# D2 — DistilGPT-2 / SST-2: live training output in Colab

Shows each stage, per-100-batch loss, running training accuracy, elapsed time, ETA, GPU memory, and end-of-epoch validation accuracy. Also writes a persistent Drive progress log.


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
!python3.12 --version
!apt-get update -qq
!apt-get install -y -qq python3.12-venv python3.12-dev
import os
if not os.path.exists('/content/ghostdp_env/bin/python'):
    !python3.12 -m venv /content/ghostdp_env


Python 3.12.3
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package python3-pip-whl.
(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack .../python3-pip-whl_24.0+dfsg-1ubuntu1.3_all.deb ...
Unpacking python3-pip-whl (24.0+dfsg-1ubuntu1.3) ...
Selecting previously unselected package python3-setuptools-whl.
Preparing to unpack .../python3-setuptools-whl_68.1.2-2ubuntu1.2_all.deb ...
Unpacking python3-setuptools-whl (68.1.2-2ubuntu1.2) ...
Selecting previously unselected package python3.12-dev.
Preparing to unpack .../python3.12-dev_3.12.3-1ubuntu0.17_amd64.deb ...
Unpacking python3.12-dev (3.12.3-1ubuntu0.17) ...
Selecting previously unselected package python3.12-venv.
Preparing to unpack .../python3.12-venv_3.12.3-1ubuntu0.17_amd64.deb ...
Unpacking python3.12-venv (3.12.3-1

In [4]:
!/content/ghostdp_env/bin/python -m pip install -q 'torch==2.6.0' 'torchvision==0.21.0' 'torchaudio==2.6.0' --index-url https://download.pytorch.org/whl/cu124


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 85.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 222.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 87.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 59.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 65.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 73.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 69.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 66.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.1/150.1 MB 68.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 67.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 103.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
!/content/ghostdp_env/bin/python -m pip install -q 'transformers==4.44.2' 'tokenizers==0.19.1' 'huggingface_hub==0.24.7' 'datasets==2.21.0' 'numpy==1.26.4' 'scipy==1.12.0' 'psutil' 'tqdm' 'opt_einsum' 'prv-accountant'
!/content/ghostdp_env/bin/python -m pip install -q --no-deps git+https://github.com/lxuechen/private-transformers.git
!/content/ghostdp_env/bin/python -m pip install -q --no-deps git+https://github.com/lxuechen/ml-swissknife.git


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.6/111.6 kB 10.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 103.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 117.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.5/417.5 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [8]:
import subprocess
# Install missing dependencies 'click', 'spacy' and downgrade pyarrow to make it compatible with NumPy 1.x
subprocess.run(['/content/ghostdp_env/bin/python', '-m', 'pip', 'install', '-q', 'click', 'spacy', 'pyarrow<15.0.0'], check=True)

code='''import torch, transformers, datasets, numpy, scipy, spacy\nfrom private_transformers import PrivacyEngine\nprint("GPU:",torch.cuda.get_device_name(0))\nprint("Transformers:",transformers.__version__)\nprint("PrivacyEngine: OK")''' if 'D3' == 'D3' else '''import torch, transformers, datasets, numpy, scipy\nfrom private_transformers import PrivacyEngine\nprint("GPU:",torch.cuda.get_device_name(0))\nprint("Transformers:",transformers.__version__)\nprint("PrivacyEngine: OK")'''

try:
    subprocess.run(['/content/ghostdp_env/bin/python', '-c', code], check=True, capture_output=True, text=True)
    print("Subprocess completed successfully.")
except subprocess.CalledProcessError as e:
    print("Subprocess failed with the following output:")
    print("STDOUT:", e.stdout)
    print("STDERR:", e.stderr)
    raise e

Subprocess completed successfully.


## Seeds and output directory
Change `SEEDS` below as needed. Each seed writes to its own Google Drive folder. Avoid running multiple notebooks simultaneously on the same GPU.


In [9]:
SEEDS = [40, 50, 60, 6, 37]
OUTPUT_DIR = '/content/drive/MyDrive/DualTune_GhostDP/DistilGPT2/multiseed'


In [10]:
from pathlib import Path
script_path = Path('/content/D2_DistilGPT2_SST2_multiseed.py')
script_path.write_text('#!/usr/bin/env python3\n"""DistilGPT-2 SST-2 reproducible experiment. Run under the Python 3.12 Colab venv."""\nimport argparse, os, time, random, json, csv, gc\nfrom pathlib import Path\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader\nfrom transformers import GPT2TokenizerFast, GPT2LMHeadModel\nfrom datasets import load_dataset, Dataset\nfrom private_transformers import PrivacyEngine\nfrom tqdm.auto import tqdm\nimport psutil\nfrom datetime import datetime\n\nLOG_FILE = None\n\ndef log(message):\n    stamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")\n    line = f"[{stamp}] {message}"\n    print(line, flush=True)\n    if LOG_FILE is not None:\n        with open(LOG_FILE, "a", encoding="utf-8") as f:\n            f.write(line + "\\n")\n\ndef progress(seed, epoch, step, total, loss, started, train_acc):\n    done = step + 1\n    elapsed = time.monotonic() - started\n    eta = elapsed / done * (total - done)\n    mem = torch.cuda.memory_allocated() / 2**20\n    log(f"Seed {seed} | DP epoch {epoch}/{DP_EPOCHS} | "\n        f"{done}/{total} batches ({done/total:.1%}) | "\n        f"loss={loss:.4f} | TrainAcc={train_acc:.4f} | elapsed={elapsed/60:.1f} min | "\n        f"ETA={eta/60:.1f} min | GPU allocated={mem:.0f} MB")\n\n\nMODEL_NAME=\'distilgpt2\'; MAX_LEN=128; BATCH_SIZE=24\nPHASE1_EPOCHS=5; DP_EPOCHS=10; TARGET_EPSILON=3.0\nMAX_GRAD_NORM=0.05; LR_P1=1e-4; LR_DP=1e-4\nPOS_STR=\' positive\'; NEG_STR=\' negative\'\n\ndef seed_all(seed):\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.deterministic=True\n    torch.backends.cudnn.benchmark=False\n\ndef balance(ds,seed):\n    # Exact Python random.sample sequence from original uploaded notebook.\n    a=[ex for ex in ds if ex[\'label\']==0]\n    b=[ex for ex in ds if ex[\'label\']==1]\n    random.seed(seed)\n    n=min(len(a),len(b))\n    chosen=random.sample(a,n)+random.sample(b,n)\n    random.shuffle(chosen)\n    return Dataset.from_list(chosen)\n\ndef collate(batch):\n    return {k:torch.tensor([x[k] for x in batch],dtype=torch.long)\n            for k in (\'input_ids\',\'attention_mask\',\'label_token\')}\n\ndef last_logits(model,ids,mask):\n    logits=model(input_ids=ids,attention_mask=mask).logits\n    return logits[torch.arange(ids.shape[0],device=ids.device),\n                  (mask.sum(1)-1).clamp(min=0),:]\n\ndef eval_acc(model,loader,device,pos,neg):\n    model.eval(); good=total=0\n    with torch.no_grad():\n        for batch in loader:\n            ids=batch[\'input_ids\'].to(device); mask=batch[\'attention_mask\'].to(device)\n            target=batch[\'label_token\'].to(device)\n            logits=last_logits(model,ids,mask)\n            pred=torch.where(logits[:,pos]>=logits[:,neg],pos,neg)\n            good+=(pred==target).sum().item(); total+=len(target)\n    model.train()\n    return good/total\n\ndef setup_data(seed):\n    raw=load_dataset(\'nyu-mll/glue\',\'sst2\')\n    train=balance(raw[\'train\'],seed); val=balance(raw[\'validation\'],seed)\n    tok=GPT2TokenizerFast.from_pretrained(MODEL_NAME)\n    tok.pad_token=tok.eos_token\n    pos=tok.encode(POS_STR,add_special_tokens=False)\n    neg=tok.encode(NEG_STR,add_special_tokens=False)\n    assert len(pos)==len(neg)==1\n    pos,neg=pos[0],neg[0]\n    def encode(ex):\n        encoded=tok(f"Review: {ex[\'sentence\']}\\nSentiment:",padding=\'max_length\',\n                    truncation=True,max_length=MAX_LEN)\n        return {\'input_ids\':encoded[\'input_ids\'],\n                \'attention_mask\':encoded[\'attention_mask\'],\n                \'label_token\':pos if ex[\'label\']==1 else neg}\n    val_tok=val.map(encode)\n    val_loader=DataLoader(val_tok,batch_size=BATCH_SIZE,shuffle=False,collate_fn=collate)\n    return train,encode,val_loader,tok,pos,neg\n\ndef build_model(tok,weight_tying,device):\n    m=GPT2LMHeadModel.from_pretrained(MODEL_NAME)\n    m.config.pad_token_id=tok.pad_token_id\n    if weight_tying:\n        m.tie_weights()\n    else:\n        m.lm_head.weight=nn.Parameter(m.lm_head.weight.detach().clone())\n    m.to(device)\n    tied=(m.lm_head.weight.data_ptr()==m.transformer.wte.weight.data_ptr())\n    assert tied==weight_tying,f\'WT check failed: {tied}\'\n    return m\n\ndef resource_start():\n    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()\n    return time.time(),psutil.Process().memory_info().rss/2**20\n\ndef resource_end(start):\n    t,rss=start\n    return dict(time_min=(time.time()-t)/60,\n                gpu_peak_allocated_mb=torch.cuda.max_memory_allocated()/2**20,\n                gpu_peak_reserved_mb=torch.cuda.max_memory_reserved()/2**20,\n                cpu_rss_delta_mb=psutil.Process().memory_info().rss/2**20-rss)\n\ndef train_dp(model,train_tok,val_loader,device,pos,neg,clipping,seed):\n    loader=DataLoader(train_tok,batch_size=BATCH_SIZE,shuffle=True,drop_last=True,collate_fn=collate)\n    opt=torch.optim.Adam(model.parameters(),lr=LR_DP)\n    pe=PrivacyEngine(model,batch_size=BATCH_SIZE,sample_size=len(train_tok),\n                     epochs=DP_EPOCHS,max_grad_norm=MAX_GRAD_NORM,\n                     target_epsilon=TARGET_EPSILON,clipping_mode=clipping)\n    pe.attach(opt)\n    log(f"Seed {seed} | PrivacyEngine attached | clipping={clipping} | {len(loader)} batches/epoch")\n    results=[]\n    for epoch in range(DP_EPOCHS):\n        model.train(); total_loss=0; train_correct=0; train_seen=0\n        epoch_start=time.monotonic()\n        log(f"Seed {seed} | START DP epoch {epoch+1}/{DP_EPOCHS}")\n        for step,batch in enumerate(loader):\n            ids=batch[\'input_ids\'].to(device); mask=batch[\'attention_mask\'].to(device)\n            target=batch[\'label_token\'].to(device)\n            opt.zero_grad()\n            logits=last_logits(model,ids,mask)\n            loss=F.cross_entropy(logits,target,reduction=\'none\')\n            pred=torch.where(logits[:,pos]>=logits[:,neg],pos,neg)\n            train_correct+=(pred.detach()==target).sum().item()\n            train_seen+=target.numel()\n            opt.step(loss=loss)\n            total_loss+=loss.detach().mean().item()\n            if step==0 or (step+1)%100==0 or step+1==len(loader):\n                progress(seed,epoch+1,step,len(loader),loss.mean().item(),epoch_start,train_correct/train_seen)\n        acc=eval_acc(model,val_loader,device,pos,neg)\n        log(f\'Seed {seed} | END DP epoch {epoch+1} | mean_loss={total_loss/len(loader):.4f} | TrainAcc={train_correct/train_seen:.4f} | val_acc={acc:.4f} | elapsed={(time.monotonic()-epoch_start)/60:.1f} min\')\n        results.append({\'epoch\':epoch+1,\'loss\':total_loss/len(loader),\'validation_accuracy\':acc})\n    # Do not pass epoch count as `steps`; use actual optimizer steps for privacy accounting.\n    spent=None\n    try:\n        spent=pe.get_privacy_spent(steps=DP_EPOCHS*len(loader))\n    except Exception as e:\n        log(f\'Seed {seed} | Privacy accountant query failed: {e!r}\')\n    return results,repr(spent)\n\ndef save_result(args,config,seed,model,metrics,epochs,privacy):\n    out=Path(args.output_dir)/config/f\'seed_{seed}\';out.mkdir(parents=True,exist_ok=True)\n    torch.save(model.state_dict(),out/\'final_model.pth\')\n    record={\'configuration\':config,\'seed\':seed,\'gpu\':torch.cuda.get_device_name(0),\n            \'target_epsilon\':TARGET_EPSILON,\'max_grad_norm\':MAX_GRAD_NORM,\n            \'batch_size\':BATCH_SIZE,\'dp_epochs\':DP_EPOCHS,\'privacy_report\':privacy,\n            \'validation_accuracy\':epochs[-1][\'validation_accuracy\'],**metrics,\n            \'epoch_metrics\':epochs}\n    (out/\'results.json\').write_text(json.dumps(record,indent=2))\n    with (Path(args.output_dir)/f\'{config}_summary.csv\').open(\'a\',newline=\'\') as f:\n        row={k:v for k,v in record.items() if k!=\'epoch_metrics\'}\n        writer=csv.DictWriter(f,fieldnames=list(row))\n        if f.tell()==0:writer.writeheader()\n        writer.writerow(row)\n    log(f\'Seed {seed} | SAVED {out}\')\n\ndef parse():\n    p=argparse.ArgumentParser()\n    p.add_argument(\'--seeds\',nargs=\'+\',type=int,default=[60])\n    p.add_argument(\'--output-dir\',default=\'/content/drive/MyDrive/DualTune_GhostDP/multiseed\')\n    return p.parse_args()\n\n\ndef main():\n    args=parse();Path(args.output_dir).mkdir(parents=True,exist_ok=True)\n    assert torch.cuda.is_available(),\'Colab GPU required\'\n    device=torch.device(\'cuda\')\n    global LOG_FILE\n    LOG_FILE=Path(args.output_dir)/\'D2_progress.log\'\n    log(f\'CONFIG D2 | WT=False | clipping=ghost | GPU={torch.cuda.get_device_name(0)} | seeds={args.seeds}\')\n    for seed in args.seeds:\n        log(f\'===== START SEED {seed} =====\')\n        seed_start=time.monotonic()\n        seed_all(seed)\n        log(f\'Seed {seed} | Loading, balancing, and tokenizing SST-2\')\n        train,encode,val_loader,tok,pos,neg=setup_data(seed)\n        log(f\'Seed {seed} | Dataset ready: {len(train)} training examples\')\n        out=Path(args.output_dir)/\'D2\'/f\'seed_{seed}\'\n        out.mkdir(parents=True,exist_ok=True)\n        log(f\'Seed {seed} | Loading DistilGPT-2 (WT=False)\')\n        model=build_model(tok,False,device)\n        log(f\'Seed {seed} | Tokenizing training examples\')\n        train_tok=train.map(encode)\n        log(f\'Seed {seed} | Training dataset tokenized\')\n        start=resource_start()\n        epochs,privacy=train_dp(model,train_tok,val_loader,device,pos,neg,\'ghost\',seed)\n        metrics=resource_end(start)\n        log(f\'Seed {seed} | Saving final model and results\')\n        save_result(args,\'D2\',seed,model,metrics,epochs,privacy)\n        log(f\'Seed {seed} | COMPLETE | total elapsed={(time.monotonic()-seed_start)/60:.1f} min\')\n        del model,train_tok,train,val_loader,tok;gc.collect();torch.cuda.empty_cache()\n\nif __name__==\'__main__\':main()\n')
print('Wrote:', script_path, '| bytes:', script_path.stat().st_size)


Wrote: /content/D2_DistilGPT2_SST2_multiseed.py | bytes: 9615


In [11]:
import subprocess
import sys
cmd = ['/content/ghostdp_env/bin/python', '-u', str(script_path), '--seeds', *map(str, SEEDS), '--output-dir', OUTPUT_DIR]
print('Starting:', ' '.join(cmd), flush=True)
# Forward child output immediately into this same Colab cell.
with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
    for line in proc.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
    returncode = proc.wait()
if returncode:
    raise RuntimeError(f'Experiment failed with exit code {returncode}')


Starting: /content/ghostdp_env/bin/python -u /content/D2_DistilGPT2_SST2_multiseed.py --seeds 40 50 60 6 37 --output-dir /content/drive/MyDrive/DualTune_GhostDP/DistilGPT2/multiseed
[2026-10-09 10:24:17] CONFIG D2 | WT=False | clipping=ghost | GPU=NVIDIA L4 | seeds=[40, 50, 60, 6, 37]
[2026-10-09 10:24:18] ===== START SEED 40 =====
[2026-10-09 10:24:18] Seed 40 | Loading, balancing, and tokenizing SST-2





Generating train split: 100%|██████████| 67349/67349 [00:00<00:00, 1372165.29 examples/s]

Generating validation split: 100%|██████████| 872/872 [00:00<00:00, 362840.58 examples/s]

Generating test split: 100%|██████████| 1821/1821 [00:00<00:00, 542151.30 examples/s]
/content/ghostdp_env/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this i

In [12]:
from pathlib import Path
import pandas as pd
p=Path(OUTPUT_DIR)/'D2_summary.csv'
print('Summary:',p,'Exists:',p.exists())
if p.exists():
    display(pd.read_csv(p))


Summary: /content/drive/MyDrive/DualTune_GhostDP/DistilGPT2/multiseed/D2_summary.csv Exists: True


,configuration,seed,gpu,target_epsilon,max_grad_norm,batch_size,dp_epochs,privacy_report,validation_accuracy,time_min,gpu_peak_allocated_mb,gpu_peak_reserved_mb,cpu_rss_delta_mb
0,D2,40,NVIDIA L4,3.0,0.05,24,10,"{'eps_rdp': 2.987777225985462, 'alpha_rdp': 4.7}",0.795561,159.768482,4158.241211,4494.0,675.656250
1,D2,50,NVIDIA L4,3.0,0.05,24,10,"{'eps_rdp': 2.987777225985462, 'alpha_rdp': 4.7}",0.797897,160.048562,4157.741211,4514.0,-3.339844
2,D2,60,NVIDIA L4,3.0,0.05,24,10,"{'eps_rdp': 2.987777225985462, 'alpha_rdp': 4.7}",0.799065,160.044771,4157.741211,4514.0,-3.820312
3,D2,6,NVIDIA L4,3.0,0.05,24,10,"{'eps_rdp': 2.987777225985462, 'alpha_rdp': 4.7}",0.802570,160.030970,4157.741211,4514.0,-2.671875
4,D2,37,NVIDIA L4,3.0,0.05,24,10,"{'eps_rdp': 2.987777225985462, 'alpha_rdp': 4.7}",0.802570,160.068892,4157.741211,4514.0,-2.972656


In [13]:
from pathlib import Path
log_path=Path(OUTPUT_DIR)/'D2_progress.log'
print('Log:',log_path)
if log_path.exists():
    print(''.join(log_path.read_text().splitlines(keepends=True)[-40:]))


Log: /content/drive/MyDrive/DualTune_GhostDP/DistilGPT2/multiseed/D2_progress.log
[2026-10-09 23:26:24] Seed 37 | DP epoch 9/10 | 1800/2481 batches (72.6%) | loss=0.8304 | TrainAcc=0.8299 | elapsed=11.6 min | ETA=4.4 min | GPU allocated=1873 MB
[2026-10-09 23:27:02] Seed 37 | DP epoch 9/10 | 1900/2481 batches (76.6%) | loss=2.2124 | TrainAcc=0.8295 | elapsed=12.2 min | ETA=3.7 min | GPU allocated=1873 MB
[2026-10-09 23:27:41] Seed 37 | DP epoch 9/10 | 2000/2481 batches (80.6%) | loss=0.9032 | TrainAcc=0.8295 | elapsed=12.9 min | ETA=3.1 min | GPU allocated=1873 MB
[2026-10-09 23:28:19] Seed 37 | DP epoch 9/10 | 2100/2481 batches (84.6%) | loss=2.0293 | TrainAcc=0.8291 | elapsed=13.5 min | ETA=2.5 min | GPU allocated=1873 MB
[2026-10-09 23:28:58] Seed 37 | DP epoch 9/10 | 2200/2481 batches (88.7%) | loss=1.4806 | TrainAcc=0.8294 | elapsed=14.2 min | ETA=1.8 min | GPU allocated=1873 MB
[2026-10-09 23:29:37] Seed 37 | DP epoch 9/10 | 2300/2481 batches (92.7%) | loss=2.3752 | TrainAcc=0.82